In [14]:
from collections import defaultdict
from pathlib import Path

import chromadb
from chromadb.errors import NotFoundError
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.schema import NodeWithScore, TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.vector_stores.chroma import ChromaVectorStore

resource module not available on Windows


In [15]:
PROJECT_ROOT = Path(r"C:\Program Files\Studying\coding\RAG_project")
PDF_DIR = PROJECT_ROOT / "data" / "wikipedia"
CHROMA_DIR = PROJECT_ROOT / "chromadb_store"
COLLECTION_NAME = "mini_wikipedia"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"

if not PDF_DIR.exists():
    raise FileNotFoundError(f"PDF folder not found: {PDF_DIR}")

# LlamaIndex uses Settings as global defaults for later index and query operations.
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)
Settings.llm = None

print(f"PDF source folder: {PDF_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Embedding model: {EMBED_MODEL_NAME}")

LLM is explicitly disabled. Using MockLLM.
PDF source folder: C:\Program Files\Studying\coding\RAG_project\data\wikipedia
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Embedding model: BAAI/bge-small-en-v1.5


In [16]:
def get_chroma_collection(
    chroma_dir: Path,
    collection_name: str,
    reset_collection: bool = False,
):
    """Create or load a persistent ChromaDB collection."""
    # PersistentClient writes vectors and metadata to disk instead of keeping them in memory.
    chroma_dir.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(chroma_dir))

    # Some ChromaDB versions raise NotFoundError when the collection is missing,
    # while older versions may raise ValueError. Catch both so reset stays safe.
    if reset_collection:
        try:
            client.delete_collection(collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except (NotFoundError, ValueError):
            print(f"Collection did not exist yet: {collection_name}")

    return client.get_or_create_collection(collection_name)


In [23]:
def load_existing_index(
    chroma_dir: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
) -> VectorStoreIndex:
    """Reconnect to an existing persisted ChromaDB collection without reinserting data."""
    # This is the path you use after closing and reopening the notebook.
    collection = get_chroma_collection(chroma_dir, collection_name, reset_collection=False)
    vector_store = ChromaVectorStore(chroma_collection=collection)
    return VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=Settings.embed_model)


def load_nodes_from_chroma_collection(collection) -> list[TextNode]:
    """Rebuild TextNode objects from the persisted Chroma collection for keyword retrieval."""
    # Chroma stores text and metadata, so we can reconstruct nodes without rerunning ingestion.
    records = collection.get(include=["documents", "metadatas"])
    documents = records.get("documents", []) or []
    metadatas = records.get("metadatas", []) or []
    ids = records.get("ids", []) or []

    nodes: list[TextNode] = []
    for node_id, document_text, metadata in zip(ids, documents, metadatas):
        if not document_text:
            continue
        nodes.append(
            TextNode(
                id_=node_id,
                text=document_text,
                metadata=metadata or {},
            )
        )

    if not nodes:
        raise ValueError("No stored text nodes were found in the Chroma collection.")

    return nodes


def build_bm25_retriever(collection, top_k: int = 5) -> BM25Retriever:
    """Create a BM25 keyword retriever from the persisted Chroma text and metadata."""
    # BM25 complements embeddings by rewarding exact keyword overlap.
    nodes = load_nodes_from_chroma_collection(collection)
    return BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k)


def reciprocal_rank_fusion(
    semantic_results: list[NodeWithScore],
    keyword_results: list[NodeWithScore],
    top_k: int,
    rrf_k: int = 60,
) -> list[NodeWithScore]:
    """Fuse semantic and keyword rankings using reciprocal rank fusion."""
    # RRF combines rankings instead of raw scores, which makes it robust across retrievers.
    fused_scores: dict[str, float] = defaultdict(float)
    node_lookup: dict[str, NodeWithScore] = {}

    for results in (semantic_results, keyword_results):
        for rank, result in enumerate(results, start=1):
            node_id = result.node.node_id
            fused_scores[node_id] += 1.0 / (rrf_k + rank)

            # Keep the first full node payload so we can print the final fused results later.
            if node_id not in node_lookup:
                node_lookup[node_id] = result

    reranked_results = sorted(
        node_lookup.values(),
        key=lambda result: fused_scores[result.node.node_id],
        reverse=True,
    )

    fused_results: list[NodeWithScore] = []
    for result in reranked_results[:top_k]:
        node_id = result.node.node_id
        fused_results.append(NodeWithScore(node=result.node, score=fused_scores[node_id]))

    return fused_results


def print_ranked_results(results: list[NodeWithScore], title: str) -> None:
    """Print retrieval results with source metadata so ranking quality is easy to inspect."""
    print(f"\n{title}")
    for rank, result in enumerate(results, start=1):
        metadata = result.node.metadata
        file_name = metadata.get("file_name", "unknown file")
        page_label = metadata.get("page_label", metadata.get("page", "unknown page"))
        chunk_number = metadata.get("chunk_number", "unknown chunk")

        print(f"\n--- Result {rank} | score={result.score:.4f} ---")
        print(f"Source: {file_name} | page: {page_label} | chunk: {chunk_number}")
        print(result.node.get_content()[:1200])


def retrieve_relevant_chunks(
    query: str,
    top_k: int = 5,
    query_index: VectorStoreIndex | None = None,
    collection=None,
    bm25_retriever: BM25Retriever | None = None,
    rrf_k: int = 60,
):
    """Run semantic search, keyword search, then fuse both rankings with reciprocal rank fusion."""
    # Load the stored vector index lazily so the function still works after reopening the notebook.
    active_index = query_index if query_index is not None else load_existing_index()

    # Semantic retrieval is strong for meaning and paraphrase matches.
    semantic_retriever = active_index.as_retriever(similarity_top_k=top_k)
    semantic_results = semantic_retriever.retrieve(query)

    # BM25 retrieval is strong for exact terms, names, acronyms, and rare keywords.
    active_collection = collection if collection is not None else get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
    active_bm25_retriever = bm25_retriever if bm25_retriever is not None else build_bm25_retriever(active_collection, top_k=top_k)
    keyword_results = active_bm25_retriever.retrieve(query)

    # Reciprocal rank fusion combines the strengths of both rankings into one final list.
    fused_results = reciprocal_rank_fusion(semantic_results, keyword_results, top_k=top_k, rrf_k=rrf_k)

    # print_ranked_results(semantic_results, "Semantic Search Results")
    # print_ranked_results(keyword_results, "BM25 Keyword Search Results")
    print_ranked_results(fused_results, "Hybrid Search Results (RRF)")

    return {
        "semantic_results": semantic_results,
        "keyword_results": keyword_results,
        "hybrid_results": fused_results,
    }

In [24]:
# Run this once after reopening the notebook to reconnect to persisted storage.
collection = get_chroma_collection(CHROMA_DIR, COLLECTION_NAME)
reloaded_index = load_existing_index()
bm25_retriever = build_bm25_retriever(collection, top_k=5)

In [27]:
sample_query = "which university is the best in hong kong?"
retrieval_outputs = retrieve_relevant_chunks(
    sample_query,
    top_k=3,
    query_index=reloaded_index,
    collection=collection,
    bm25_retriever=bm25_retriever,
)

hybrid_results = retrieval_outputs["hybrid_results"]


Hybrid Search Results (RRF)

--- Result 1 | score=0.0328 ---
Source: Hong Kong.pdf | page: 40 | chunk: 435
University (in 2006), Education University of Hong Kong (in 2016), Hang Seng University of Hong
Kong (in 2018) and Saint Francis University (in 2024) all attained full university status in subsequent
years.
Overseas universities in Hong Kong often operate as branch campuses or partnerships offering
accredited degrees, such as the University of Chicago Hong Kong, University of Sunderland,
University of Wollongong in Hong Kong. These institutions, provide local and internationally
recognized qualifications.
In 2026, QS Best Student Cities ranked Hong Kong as the 17th best city for university students. 7th
in best Asia for best student cities. Noting, high scores in employer activity, desirability, and diverse
student population.
Subject Rankings
In the 2026 QS World University Rankings by Subject, Hong Kong's higher education sector
demonstrated significant global competitiveness, 

In [ ]:
# def retrieve_relevant_chunks(
#     query: str,
#     top_k: int = 5,
#     query_index: VectorStoreIndex | None = None,
# ):
#     """Retrieve the most relevant chunks for a user question."""
#     # Use the provided index when one is passed in, otherwise fall back to the current session index.
#     active_index = query_index if query_index is not None else index
#     retriever = active_index.as_retriever(similarity_top_k=top_k)
#     results = retriever.retrieve(query)

#     # Print source metadata beside each chunk so you can inspect where answers come from.
#     for rank, result in enumerate(results, start=1):
#         metadata = result.node.metadata
#         file_name = metadata.get("file_name", "unknown file")
#         page_label = metadata.get("page_label", metadata.get("page", "unknown page"))
#         chunk_number = metadata.get("chunk_number", "unknown chunk")

#         print(f"\n--- Result {rank} | score={result.score:.4f} ---")
#         print(f"Source: {file_name} | page: {page_label} | chunk: {chunk_number}")
#         print(result.node.get_content()[:1200])

#     return results